# Step 4. Transfer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/younesStrittmatter/sweetpea-starter/blob/main/tutorial/notebooks/4_transfer.ipynb)

The tutorial step without an assistant. Run the cells from top to bottom; the tables below are from one run, yours will differ where the design leaves things open.

In [ ]:
%pip install -q sweetpea pandas

**So far.** Step 2 made the trial before a factor, and step 3 made blocks that are mostly congruent or mostly incongruent. This step puts both ideas together and adds a second task.

**The question.** Does adapting to frequent conflict make it harder to switch to another task?

**Prediction if shielding and switching are one control setting.** The switch cost is larger in a mostly incongruent context than in a mostly congruent one.

**The alternative to rule out.** A mostly incongruent context is simply harder, so everything is slower there.

**The analysis.** The switch cost (switch against repeat trials) in mostly incongruent mini-blocks against mostly congruent mini-blocks.

### As before, plus a second task and the context

A cue says on each trial whether to name the ink or read the word. The task is a regular factor. So is the context now: in step 3 it was a column added by hand, here it is part of the design.

In [ ]:
import pandas as pd
from sweetpea import (Factor, DerivedLevel, WithinTrial, Transition, Exclude, MinimumTrials,
                      CrossBlock, synthesize_trials, CMSGen)

colours = ["red", "green", "blue"]

# regular factors: set freely on every trial
colour = Factor("colour", colours)      # the ink
word = Factor("word", colours)          # what the word says
from sweetpea import Nest, Repeat

task = Factor("task", ["name the colour", "read the word"])              # new: a second task, with a cue
context = Factor("context", ["mostly congruent", "mostly incongruent"])  # new: held for a whole mini-block

### New: the task transition

A transition factor, as in step 2, but on the task: does it repeat or switch? Without repeat trials in the same context there is no switch cost to measure.

In [ ]:
def task_repeats(task):
    return task[0] == task[-1]


def task_switches(task):
    return task[0] != task[-1]


transition = Factor("transition", [
    DerivedLevel("repeat", Transition(task_repeats, [task])),
    DerivedLevel("switch", Transition(task_switches, [task])),
])

### New: what the context makes frequent

In a mostly congruent mini-block the congruent trials are the frequent ones, in a mostly incongruent mini-block the incongruent ones. One derived factor says that for both: it follows from the colour, the word and the context. The third argument of a level is its weight: 3 to 1 is three quarters.

In [ ]:
def frequent_here(colour, word, context):
    congruent = colour == word
    return congruent == (context == "mostly congruent")


def rare_here(colour, word, context):
    return not frequent_here(colour, word, context)


frequency = Factor("frequency", [
    DerivedLevel("frequent", WithinTrial(frequent_here, [colour, word, context]), 3),
    DerivedLevel("rare", WithinTrial(rare_here, [colour, word, context]), 1),
])

### Something has to give

The blocks of step 3 hold 24 trials. Crossed with 2 tasks and 2 transitions that is 96 trials per mini-block: too long for a context that should change often.

So a mini-block crosses what the analysis compares: the frequency (and with it congruency), the task and the transition. That is 16 trials. The even pairs of colour and word are given up.

### New: a block inside a block

Step 3 generated every block on its own and put them one after the other. `Nest` makes it one design. The outer block crosses the context; `Repeat` asks for it twice, so four mini-blocks. The inner block is the mini-block. `Nest` holds each context for the length of one mini-block and draws the order of the contexts.

In [ ]:
contexts = Repeat(CrossBlock(design=[context], crossing=[context], constraints=[]),
                  constraints=[MinimumTrials(4)])            # four mini-blocks: each context twice
mini_block = CrossBlock(design=[colour, word, task, transition, frequency, context],
                        crossing=[frequency, task, transition],
                        constraints=[])
block = Nest(contexts, mini_block, alignment="parallel start")
session = pd.DataFrame(synthesize_trials(block, 1, CMSGen)[0])

# congruency as a column: inside a nested block, SweetPea 0.2.14 fails on a derived factor that is in no crossing
session["congruency"] = ["congruent" if colour == word else "incongruent"
                         for colour, word in zip(session["colour"], session["word"])]
session.insert(0, "block", session.index // 16 + 1)
display(pd.crosstab([session["block"], session["context"]], session["congruency"]))

,congruency,congruent,incongruent
block,context,,
1,mostly congruent,12,4
2,mostly incongruent,3,13
3,mostly incongruent,4,12
4,mostly congruent,11,5


Every mini-block holds one context, and each context comes twice.

### Where does the crossing sit?

Count the eight combinations the switch cost is computed from, in every mini-block. The first trial of the session has no transition.

In [ ]:
counted = session[session["transition"] != ""]
display(pd.crosstab([counted["block"], counted["context"]], [counted["transition"], counted["task"], counted["congruency"]]))

transition                        repeat  ...            
task                     name the colour  ...            
congruency                     congruent  ... incongruent
block context                             ...            
1     mostly congruent                 3  ...           1
2     mostly incongruent               1  ...           3
3     mostly incongruent               1  ...           3
4     mostly congruent                 3  ...           1

[4 rows x 8 columns]

Almost 3 to 1 in every mini-block, but not exactly. The reason: a transition needs a trial before, so the crossing starts at the second trial and runs one trial ahead of the context. Count again in stretches of 16 that start at the second trial:

In [ ]:
counted = counted.assign(stretch=(counted.index - 1) // 16 + 1)
display(pd.crosstab(counted["stretch"], [counted["transition"], counted["task"], counted["frequency"]]))

transition          repeat                     ...                        
task       name the colour      read the word  ...      read the word     
frequency         frequent rare      frequent  ... rare      frequent rare
stretch                                        ...                        
1                        3    1             3  ...    1             3    1
2                        3    1             3  ...    1             3    1
3                        3    1             3  ...    1             3    1
4                        3    1             2  ...    1             3    1

[4 rows x 8 columns]

There it is exact (the last stretch is one trial short). So a mini-block is the exact crossing except for one trial: its first trial, right after the context changed, belongs to the stretch before. Most analyses drop that trial anyway.

### What gave way?

The ink colour and the pairs are not in the crossing. Count them.

In [ ]:
display(pd.crosstab(session["context"], session["colour"]))
display(pd.crosstab([session["context"], session["colour"]], session["word"]))

colour,blue,green,red
context,,,
mostly congruent,14,16,2
mostly incongruent,14,11,7


word                       blue  green  red
context            colour                  
mostly congruent   blue      11      1    2
                   green      2     10    4
                   red        0      0    2
mostly incongruent blue       2      7    5
                   green      4      3    4
                   red        3      2    2

**Check.** Is there a repeat baseline in each context? How many incongruent switch trials does a mostly congruent mini-block have? Are the ink colours and the pairs still even, and does that matter for the switch cost? The first trial of a mini-block is a switch or a repeat relative to a trial from the other context: keep it or drop it?

**Try.** Put `colour` into the crossing of the mini-block. How long is a mini-block now?

**The idea.** Each question added a factor, and each factor multiplies the design. At some point not everything fits, and the design becomes a choice about what to leave open. Which choice is acceptable depends on what you compare.